# 01 — Dataset exploration and quality control

This notebook inspects **Medical Segmentation Decathlon Task01_BrainTumour** using the native 4D NIfTI layout. It does not rewrite source files.

**Source facts:** image channels are mapped through `dataset.json`; the loader returns FLAIR, T1, T1ce and T2. Original MSD labels are mapped internally to `{0,1,2,4}` for compatibility with the project's WT/TC/ET region definitions. The source dataset must be cited as MSD Task01_BrainTumour (BraTS 2016/2017-derived), not BraTS 2021.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = ROOT / "data" / "raw" / "Task01_BrainTumour"
print("Project:", ROOT)
print("Dataset:", DATA_ROOT)

In [ ]:
from qmi.data.msd_dataset import read_msd_metadata, discover_msd_cases
metadata = read_msd_metadata(DATA_ROOT)
cases = discover_msd_cases(DATA_ROOT)
print("Labelled cases:", len(cases))
print("Modality indices:", metadata.modalities)
print("Source-to-internal labels:", metadata.labels)
pd.DataFrame([{"case_id": c.case_id, "image": c.image_path.name, "label": c.label_path.name} for c in cases[:10]])

In [ ]:
from qmi.data.msd_dataset import load_msd_case, case_to_slices
case = cases[0]
volumes, masks = load_msd_case(case, metadata)
print("Case:", case.case_id)
print("Modalities (4,Z,H,W):", volumes.shape, volumes.dtype)
print("Mask (Z,H,W):", masks.shape, masks.dtype)
print("Unique internal labels:", np.unique(masks, return_counts=True))
print("Finite image values:", np.isfinite(volumes).all(), "range:", float(volumes.min()), float(volumes.max()))
print("Aligned spatial shape:", volumes.shape[1:] == masks.shape)

In [ ]:
# Choose the axial slice with the largest tumour area for a representative QC panel.
areas = np.count_nonzero(masks > 0, axis=(1,2))
z = int(np.argmax(areas))
fig, axes = plt.subplots(1, 5, figsize=(17, 4))
for ax, idx, name in zip(axes[:4], range(4), ["FLAIR", "T1", "T1ce", "T2"]):
    ax.imshow(volumes[idx, z], cmap="gray"); ax.set_title(name); ax.axis("off")
axes[4].imshow(volumes[0, z], cmap="gray")
axes[4].imshow(masks[z], cmap="nipy_spectral", alpha=.5, vmin=0, vmax=4)
axes[4].set_title(f"Ground truth overlay · z={z}"); axes[4].axis("off")
plt.tight_layout()

## Region definitions

The regions are **overlapping binary masks**, not a three-way mutually exclusive class map:

- Whole Tumour (WT): labels 1, 2 or 4
- Tumour Core (TC): labels 1 or 4
- Enhancing Tumour (ET): label 4

The test split `imagesTs` has no public labels in this dataset release. Quantitative evaluation must use a held-out patient-level subset of the labelled `imagesTr`/`labelsTr` pairs.